# Extracción de matrices de metilación por gen

Lee archivos `(ID)_(Contexto)map_(Cromosoma).txt` y genera una matriz por contexto:
- **Filas** = muestras
- **Columnas** = genes (PRUDU_ID)
- **Valores** = media del ratio de metilación en gene body (dist=0)

## 1. Configuración

In [ ]:
# Ajustar rutas y parámetros según sea necesario
METHYL_DIR  = "/"   # <-- carpeta con todos los archivos map (de macrogen)
OUTPUT_DIR  = "."                      # carpeta de salida

# IDs de muestra (los números del nombre de archivo)
# Ejemplo: 1_CGmap_Pdu1.txt ... 39_CGmap_Pdu8.txt
SAMPLE_IDS  = [s for s in range(1, 40) if s not in [3, 38]]  # excluidas: 3 (baja conversión), 38 (outlier)

# Cromosomas
CHROMS      = [f"Pdu{i}" for i in range(1, 9)]   # Pdu1 ... Pdu8

# Contextos y nombre del archivo correspondiente
CONTEXT_MAP = {
    "CG":  "CGmap",
    "CHG": "CHGmap",
    "CHH": "CHHmap",
}

# Parámetros de filtrado
MIN_COV     = 5    # cobertura mínima (CT_count)
MIN_SAMPLES = 3    # mínimo de muestras con datos para conservar un gen

print("Configuración:")
print(f"  Directorio: {METHYL_DIR}")
print(f"  Muestras:   {len(SAMPLE_IDS)} ({SAMPLE_IDS[0]}...{SAMPLE_IDS[-1]})")
print(f"  Cromosomas: {CHROMS}")
print(f"  Contextos:  {list(CONTEXT_MAP.keys())}")
print(f"  Cobertura mínima: {MIN_COV}")

## 2. Librerías

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from collections import defaultdict

print(f"Python: {sys.version}")
print(f"pandas: {pd.__version__}")
print(f"numpy:  {np.__version__}")

## 3. Verificar que los archivos existen

In [ ]:
def get_filepath(sample_id, context_label, chrom):
    """
    Construye la ruta completa de un archivo map.
    Estructura: METHYL_DIR/{id}/{id}_{ctx}/{id}_{ctx}_{chrom}.txt
    Ejemplo:    result_WGBS/1/1_CGmap/1_CGmap_Pdu1.txt
    """
    filename = f"{sample_id}_{context_label}_{chrom}.txt"
    subfolder = f"{sample_id}_{context_label}"
    return os.path.join(METHYL_DIR, str(sample_id), subfolder, filename)

# Verificar cuántos archivos existen
total_esperados   = len(SAMPLE_IDS) * len(CONTEXT_MAP) * len(CHROMS)
total_encontrados = 0
missing = []

for sid in SAMPLE_IDS:
    for ctx, ctx_label in CONTEXT_MAP.items():
        for chrom in CHROMS:
            fp = get_filepath(sid, ctx_label, chrom)
            if os.path.exists(fp):
                total_encontrados += 1
            else:
                missing.append(fp)

print(f"Archivos esperados:    {total_esperados}")
print(f"Archivos encontrados:  {total_encontrados}")
print(f"Archivos no encontrados: {len(missing)}")

if missing:
    print("\nPrimeros 10 no encontrados:")
    for f in missing[:10]:
        print(f"  {f}")
    if total_encontrados == 0:
        print("\nATENCION: Si faltan muchos archivos, revisa METHYL_DIR y el patron de nombres.")
else:
    print("\nTodos los archivos encontrados OK")


## 4. Verificar formato de un archivo

In [ ]:
# Muestra las primeras líneas del primer archivo CGmap disponible
fp_test = get_filepath(SAMPLE_IDS[0], "CGmap", CHROMS[0])
print(f"Archivo de prueba: {fp_test}\n")

if os.path.exists(fp_test):
    with open(fp_test, "r") as f:
        for i, line in enumerate(f):
            parts = line.strip().split("\t")
            print(f"Línea {i+1} ({len(parts)} cols): {line[:120].strip()}")
            if i >= 9:
                break
else:
    print("Archivo no encontrado. Revisa la ruta y el patrón de nombres.")

## 5. Funciones de extracción

In [ ]:
def process_sample(sample_id, context, context_label):
    """
    Procesa todos los cromosomas de una muestra para un contexto dado.
    
    Columnas:
    0=Chrom, 1=Pos, 2=Strand, 3=Context, 4=Methylation_ratio,
    5=eff_CT_count, 6=C_count, 7=CT_count, ..., 15=Region, 16=Gene
    
    Gene body = exonic + intronic
    Promotor  = upstream
    """
    gene_ratios_body   = defaultdict(list)
    gene_ratios_promo  = defaultdict(list)
    n_sites = 0
    n_used  = 0
    n_files = 0

    GENEBODY_REGIONS = {"exonic", "intronic"}
    PROMOTER_REGIONS = {"upstream"}

    for chrom in CHROMS:
        fp = get_filepath(sample_id, context_label, chrom)
        if not os.path.exists(fp):
            continue
        n_files += 1

        with open(fp, "r") as fh:
            for line in fh:
                parts = line.rstrip("\n").split("\t")

                if parts[0] == "Chrom":
                    continue
                if len(parts) < 17:
                    continue

                ctx    = parts[3]
                ratio  = parts[4]
                cov_tc = parts[7]
                region = parts[15]
                gene   = parts[16].strip()

                if ctx != context:
                    continue

                n_sites += 1

                # Filtrar por región de interés
                if region not in GENEBODY_REGIONS and region not in PROMOTER_REGIONS:
                    continue

                # Gen válido
                if not gene or gene == "." or not gene.startswith("PRUDU"):
                    continue

                # Filtrar cobertura
                if ratio == "." or cov_tc == ".":
                    continue
                try:
                    r   = float(ratio)
                    cov = int(cov_tc)
                except ValueError:
                    continue

                if cov < MIN_COV:
                    continue

                if region in GENEBODY_REGIONS:
                    gene_ratios_body[gene].append(r)
                else:
                    gene_ratios_promo[gene].append(r)

                n_used += 1

    body_dict  = {g: float(np.mean(v)) for g, v in gene_ratios_body.items()}
    promo_dict = {g: float(np.mean(v)) for g, v in gene_ratios_promo.items()}
    stats = (n_files, n_sites, n_used, len(body_dict), len(promo_dict))
    return body_dict, promo_dict, stats


print("Funciones definidas OK")
print(f"Gene body = exonic + intronic  |  Promotor = upstream  |  Cov >= {MIN_COV}")


## 6. Test rápido — una muestra, contexto CG

In [ ]:
# Test rápido con muestra 1, contexto CG
sid_test = SAMPLE_IDS[0]
print(f"Testeando muestra {sid_test} | contexto CG...")

body_dict, promo_dict, (n_files, n_sites, n_used, n_body, n_promo) = process_sample(
    sid_test, "CG", "CGmap"
)

print(f"  Archivos leídos:        {n_files}/8")
print(f"  Sitios CG totales:      {n_sites:,}")
print(f"  Sitios usados:          {n_used:,}")
print(f"  Genes gene body:        {n_body:,}")
print(f"  Genes promotor:         {n_promo:,}")

if n_body > 0 and n_promo > 0:
    print("\nEjemplos gene body:")
    for gid, val in list(body_dict.items())[:3]:
        print(f"  {gid}: {val:.4f}")
    print("\nEjemplos promotor:")
    for gid, val in list(promo_dict.items())[:3]:
        print(f"  {gid}: {val:.4f}")
    print("\nTEST OK — puedes lanzar la extracción completa")
else:
    print("\nATENCION: 0 genes. Revisa el formato del archivo.")


## 7. Extracción completa — 3 contextos, todas las muestras

**Esta celda tarda varias horas. Asegurarse primero de que el test de la celda 6 funciona antes de lanzarla.**

In [ ]:
import time
os.makedirs(OUTPUT_DIR, exist_ok=True)

for context, context_label in CONTEXT_MAP.items():
    print(f"\n{'='*60}")
    print(f"Contexto: {context}")
    print(f"{'='*60}")
    t0_ctx = time.time()

    matrix_body  = {}
    matrix_promo = {}

    for i, sid in enumerate(SAMPLE_IDS):
        t0 = time.time()
        body_dict, promo_dict, (n_files, n_sites, n_used, n_body, n_promo) = process_sample(
            sid, context, context_label
        )
        elapsed = time.time() - t0
        matrix_body[str(sid)]  = body_dict
        matrix_promo[str(sid)] = promo_dict
        print(f"  [{i+1:2d}/{len(SAMPLE_IDS)}] muestra {sid:2d} | "
              f"body={n_body:,} promo={n_promo:,} genes | {elapsed:.0f}s")

    for label, matrix in [("genebody", matrix_body), ("promoter", matrix_promo)]:
        df = pd.DataFrame(matrix).T
        df.index.name = "sample_id"
        valid = df.notna().sum() >= MIN_SAMPLES
        df = df.loc[:, valid]
        outfile = os.path.join(OUTPUT_DIR, f"methyl_{context}_{label}.csv")
        df.to_csv(outfile)
        print(f"  -> {outfile}  ({df.shape[0]} muestras x {df.shape[1]:,} genes)")

    print(f"  Tiempo: {(time.time()-t0_ctx)/60:.1f} min")

print("\nFINALIZADO")


## 8. Resumen de las matrices generadas

In [ ]:
for context in CONTEXT_MAP.keys():
    for label in ["genebody", "promoter"]:
        outfile = os.path.join(OUTPUT_DIR, f"methyl_{context}_{label}.csv")
        if not os.path.exists(outfile):
            print(f"{context} {label}: no generado todavía")
            continue
        df = pd.read_csv(outfile, index_col=0)
        print(f"\nmethyl_{context}_{label}.csv")
        print(f"  Muestras: {df.shape[0]}")
        print(f"  Genes:    {df.shape[1]:,}")
        print(f"  % missing: {df.isna().mean().mean()*100:.1f}%")
        print(f"  Ratio medio: {df.mean().mean():.4f}")
